# **Calculus and Matrix Calculus**

**Prerequisites:** Linear Algebra | **Next:** Probability | **Depth tier:** Foundation

## 1. Theory

Every training algorithm in this repo (Gradient Descent, Newton's method,
EM, SVM's dual) is "find the input that makes some function's output
smallest/largest." Calculus is the machinery for finding that point.
Matrix calculus extends single-variable calculus to functions of vectors - because ML models have many parameters at once ($w\in\mathbb R^d$, not
a single $w$).

## 2. Mathematical Derivation - the four objects you'll use constantly

**Derivative** (single variable): $f'(x)=\lim_{h\to0}\frac{f(x+h)-f(x)}{h}$.

**Gradient** (scalar function of a vector, $f:\mathbb R^d\to\mathbb R$):
$$\nabla f(\mathbf x) = \left(\frac{\partial f}{\partial x_1},\dots,\frac{\partial f}{\partial x_d}\right)$$
Points in the direction of steepest **increase** of $f$ — the entire
justification for gradient *descent* moving in $-\nabla f$
(`06_Optimization` derives this precisely).

**Jacobian** (vector-valued function of a vector, $\mathbf f:\mathbb R^d\to\mathbb R^m$):
$$J_{ij} = \frac{\partial f_i}{\partial x_j}, \qquad J\in\mathbb R^{m\times d}$$
The gradient is a special case: a Jacobian where $m=1$ (row vector,
usually written as a column by convention).

**Hessian** (second derivatives of a scalar function):
$$H_{ij} = \frac{\partial^2f}{\partial x_i\partial x_j}, \qquad H\in\mathbb R^{d\times d},\ H=H^T$$
Already used without derivation in `06_Optimization/02_convexity.ipynb` —
$H\succeq0$ (positive semi-definite) everywhere ⟺ $f$ convex. Also
used in Newton's method (`03_Supervised_Learning/Ensemble_Learning/07_xgboost_lightgbm_catboost.ipynb`'s XGBoost note on
second-order boosting uses exactly this).

**Chain rule** (vector form) — the single most-used rule in all of ML:
$$\frac{\partial f(\mathbf g(\mathbf x))}{\partial \mathbf x} = \left(\frac{\partial \mathbf g}{\partial \mathbf x}\right)^T \frac{\partial f}{\partial \mathbf g}$$
This is exactly how Logistic Regression's gradient was derived in
`03_Supervised_Learning\Classification\01_logistic_regression.ipynb` §4: $J$ depends on $x$ only through
$z=w^Tx+b$, so $\frac{\partial J}{\partial w}=\frac{\partial J}{\partial p}\cdot\frac{\partial p}{\partial z}\cdot\frac{\partial z}{\partial w}$
— chaining three simple derivatives together instead of one hard one.

## 3. Worked Numerical Example

$f(x_1,x_2) = x_1^2x_2 + 3x_2^2$ at point $(2,1)$.

**Gradient**: $\frac{\partial f}{\partial x_1}=2x_1x_2=2(2)(1)=4$;
$\frac{\partial f}{\partial x_2}=x_1^2+6x_2=4+6=10$. So
$\nabla f(2,1) = (4,10)$.

**Hessian**: $\frac{\partial^2f}{\partial x_1^2}=2x_2=2$;
$\frac{\partial^2f}{\partial x_1\partial x_2}=2x_1=4$;
$\frac{\partial^2f}{\partial x_2^2}=6$.
$$H = \begin{bmatrix}2&4\\4&6\end{bmatrix}$$
Eigenvalues of $H$: $\det(H-\lambda I)=(2-\lambda)(6-\lambda)-16=\lambda^2-8\lambda-4=0$,
$\lambda = \frac{8\pm\sqrt{64+16}}{2}=\frac{8\pm8.944}{2}$ → $\lambda_1\approx8.47,\ \lambda_2\approx-0.47$.
One negative eigenvalue → $H$ is **not** PSD at this point → $f$ is **not
convex** at $(2,1)$ (it's a saddle-point region) — a concrete illustration
of using the Hessian to check convexity, not just citing the rule.

In [1]:
## numerical differentiation as a sanity check

import numpy as np

def f(x): return x[0]**2 * x[1] + 3*x[1]**2

def numerical_gradient(f, x, h=1e-6):
    grad = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += h
        x_minus = x.copy(); x_minus[i] -= h
        grad[i] = (f(x_plus) - f(x_minus)) / (2*h)
    return grad

x0 = np.array([2.0, 1.0])
print(numerical_gradient(f, x0))   # ~[4.0, 10.0] -- matches hand calc

[ 4. 10.]


## 4. Visual Intuition

![gradient descent contours](../_assets/gradient_descent_convergence.png)

(already generated for `31-Optimization-Section.md`; the contour lines are
literally level sets of a cost function, and the gradient at any point is
perpendicular to the contour through that point — worth stating this
geometric fact explicitly when reusing the figure here.)

In [2]:
## Practical Implementation — symbolic differentiation

import sympy as sp

x1, x2 = sp.symbols('x1 x2')
f = x1**2*x2 + 3*x2**2
grad = [sp.diff(f, v) for v in (x1, x2)]
hess = sp.hessian(f, (x1, x2))
print(grad)   # [2*x1*x2, x1**2 + 6*x2]
print(hess.subs({x1:2, x2:1}))   # Matrix([[2, 4], [4, 6]]) -- matches hand calc

[2*x1*x2, x1**2 + 6*x2]
Matrix([[2, 4], [4, 6]])


## 5. Failure Cases

Gradient descent on a non-convex function (like this $f$ in some regions)
can converge to a saddle point or local minimum rather than the global
one — the Hessian eigenvalue check in §3 is exactly how you'd diagnose
this rather than just observing "training stalled."

## 6. Assumptions

Differentiability (many ML loss functions, e.g. hinge loss, ReLU, are
non-differentiable at specific points — **subgradients** handle this,
briefly mentioned in `31-Optimization-Section.md` and revisited if a
subgradient methods note is added later).

## 7. Complexity

Computing a full Hessian for $d$ parameters costs $O(d^2)$ storage and is
often $O(d^2)$–$O(d^3)$ to compute/invert (Newton's method) — the
practical reason first-order methods (plain gradient descent, Adam) that
only need $O(d)$-cost gradients dominate in high-dimensional ML (large
$d$) despite Newton's faster per-step convergence.